In [66]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler


In [67]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [68]:
X_train_scaled

array([[ 0.16300505, -1.22997086,  0.26061204, ..., -0.03549922,
        -0.1139106 ,  0.47221789],
       [ 2.49660303,  0.14642806,  2.43498662, ...,  1.71294967,
         0.0259633 , -0.60068043],
       [ 0.86030635,  1.24990001,  0.90549893, ...,  1.16088752,
         0.72533281,  2.79581228],
       ...,
       [-0.60374857, -1.95228619, -0.6304457 , ..., -0.67135381,
        -0.74766026, -0.6090147 ],
       [-0.28426789, -0.82293323, -0.24746844, ..., -0.6097951 ,
         0.50257067, -0.09729054],
       [-1.24937734, -0.55706472, -1.2156059 , ..., -0.48319893,
         0.15374933,  0.81503418]])

In [69]:
y_test

array([0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1,
       0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1])

In [70]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()

y_train_tensor = torch.from_numpy(y_train).unsqueeze(1).float()
y_test_tensor = torch.from_numpy(y_test).unsqueeze(1).float()

In [71]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)


In [72]:
X_train_scaled_tensor.shape

torch.Size([455, 30])

In [73]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define Our Neural Network

In [74]:
class BCNet(nn.Module):
  def __init__(self):
    super(BCNet, self).__init__()

    self.fc1 = nn.Linear(30, 64)
    self.fc2 = nn.Linear(64, 32)
    self.fc3 = nn.Linear(32, 1)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.sigmoid(self.fc3(x))
    return x

In [75]:
model = BCNet()


In [76]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [77]:
epochs = 20

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for x_batch, y_batch in train_loader:
    optimizer.zero_grad()
    y_pred = model(x_batch)
    loss = criterion(y_pred, y_batch)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader)}")

Epoch 1/20 - Loss: 0.6442724863688151
Epoch 2/20 - Loss: 0.5195508003234863
Epoch 3/20 - Loss: 0.3718049506346385
Epoch 4/20 - Loss: 0.23552882969379424
Epoch 5/20 - Loss: 0.15139420280853907
Epoch 6/20 - Loss: 0.11343445728222529
Epoch 7/20 - Loss: 0.08734151348471642
Epoch 8/20 - Loss: 0.0792609415948391
Epoch 9/20 - Loss: 0.07274201897283396
Epoch 10/20 - Loss: 0.06388307126859824
Epoch 11/20 - Loss: 0.07917477923134962
Epoch 12/20 - Loss: 0.06381945908069611
Epoch 13/20 - Loss: 0.06427014575650294
Epoch 14/20 - Loss: 0.05094915533748766
Epoch 15/20 - Loss: 0.048746891071399054
Epoch 16/20 - Loss: 0.046388665400445464
Epoch 17/20 - Loss: 0.044155706620464724
Epoch 18/20 - Loss: 0.04260180915395419
Epoch 19/20 - Loss: 0.04197131688706577
Epoch 20/20 - Loss: 0.040646193611125155


In [79]:
with torch.no_grad():
  model.eval()
  y_pred = model(X_test_scaled_tensor)
  loss = criterion(y_pred, y_test_tensor).item()
  accuracy = ((y_pred >= 0.5) == y_test_tensor).float().mean().item()

In [80]:
accuracy

1.0